In [4]:
import os
import json
import requests
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# =========================================================================
# Configuration
# =========================================================================

PATH          = r'C:\Users\lidon\Desktop\2026spring\IDX_MLS_Analytics'
SOLD_CSV      = os.path.join(PATH, 'all_sold_cleaned.csv')
LISTING_CSV   = os.path.join(PATH, 'all_listings_cleaned.csv')
DTYPE_SPEC    = {'PostalCode': str, 'ListingKey': str}

SOLD_OUTPUT   = os.path.join(PATH, 'sold_residential_features_enriched.csv')
LISTING_OUTPUT= os.path.join(PATH, 'list_residential_features_enriched.csv')

DISTRICT_URL        = "https://gis.data.ca.gov/api/download/v1/items/48870daecfe14c6ab376f6a673491914/geojson?layers=0"
DISTRICT_CACHE_PATH = os.path.join(PATH, 'ca_unified_school_districts_2025_26.geojson')

DATE_COLS = ['CloseDate', 'PurchaseContractDate',
             'ListingContractDate', 'ContractStatusChangeDate']

# =========================================================================
# Helper: section header
# =========================================================================

def section(title):
    print()
    print("=" * 78)
    print(title)
    print("=" * 78)

# =========================================================================
# Helper: safe_divide (from teammate 1)
# Treats denominator <= 0 or NaN as invalid → returns NaN
# More concise than np.where and handles edge cases correctly
# =========================================================================

def safe_divide(numerator, denominator):
    valid_denom = denominator.mask(denominator <= 0)
    return numerator / valid_denom

# =========================================================================
# Load datasets
# =========================================================================

section("Load Cleaned Data")

sold_raw     = pd.read_csv(SOLD_CSV,    dtype=DTYPE_SPEC, low_memory=False)
listings_raw = pd.read_csv(LISTING_CSV, dtype=DTYPE_SPEC, low_memory=False)

sold     = sold_raw.copy()
listings = listings_raw.copy()

for df in [sold, listings]:
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"Sold    : {len(sold):,} rows  |  {sold.shape[1]} columns")
print(f"Listing : {len(listings):,} rows  |  {listings.shape[1]} columns")

# =========================================================================
# Part A — School District Enrichment
# Auto-downloads GeoJSON with retry logic and local cache (from teammate 1)
# =========================================================================

section("Part A: School District Enrichment")

def load_unified_districts():
    """Download (or load from cache) Unified school district boundaries."""
    if os.path.exists(DISTRICT_CACHE_PATH):
        print(f"Loading cached Unified district boundaries...")
        unified = gpd.read_file(DISTRICT_CACHE_PATH)
        print(f"Loaded {len(unified)} cached Unified district polygons.")
        return unified

    print("No cache found — downloading CA School District boundaries...")
    raw_path    = os.path.join(PATH, 'ca_school_districts_raw.geojson')
    max_attempts = 3

    for attempt in range(1, max_attempts + 1):
        try:
            print(f"  Attempt {attempt}/{max_attempts}...")
            r = requests.get(DISTRICT_URL, timeout=180)
            r.raise_for_status()
            with open(raw_path, 'wb') as f:
                f.write(r.content)
            # Verify the download is complete and valid JSON before parsing
            with open(raw_path, 'r', encoding='utf-8') as f:
                json.load(f)
            print(f"  Downloaded {os.path.getsize(raw_path)/1e6:.1f} MB successfully.")
            break
        except (requests.exceptions.RequestException, json.JSONDecodeError) as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt == max_attempts:
                raise SystemExit(
                    f"Could not download GeoJSON after {max_attempts} attempts.\n"
                    f"Download manually from:\n  {DISTRICT_URL}\n"
                    f"and save to:\n  {raw_path}"
                )

    districts = gpd.read_file(raw_path)
    print(f"Downloaded {len(districts)} total district polygons. CRS: {districts.crs}")

    if districts.crs is None:
        districts = districts.set_crs(epsg=4326)
    elif districts.crs.to_epsg() != 4326:
        print(f"Reprojecting from {districts.crs} to EPSG:4326...")
        districts = districts.to_crs(epsg=4326)

    unified = districts[districts['DistrictType'] == 'Unified'].copy()
    print(f"Filtered to {len(unified)} Unified districts (out of {len(districts)} total).")
    unified = unified[['DistrictName', 'DistrictType', 'geometry']]
    unified.to_file(DISTRICT_CACHE_PATH, driver='GeoJSON')
    print(f"Cached to: {DISTRICT_CACHE_PATH}")
    return unified


def add_district_name(df, label, unified_districts):
    """Point-in-polygon spatial join to assign DistrictName to each property."""
    if 'Latitude' not in df.columns or 'Longitude' not in df.columns:
        print(f"  [{label}] Lat/Lon not found — DistrictName set to NaN.")
        df = df.copy()
        df['DistrictName'] = np.nan
        return df

    points = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
        crs='EPSG:4326',
    )
    joined = gpd.sjoin(
        points,
        unified_districts[['DistrictName', 'geometry']],
        how='left',
        predicate='within',
    )
    # Handle multi-match from boundary overlaps — keep first match per property
    multi = int(joined.index.duplicated(keep='first').sum())
    if multi > 0:
        print(f"  [{label}] {multi} properties matched >1 district — kept first match.")
    joined = joined[~joined.index.duplicated(keep='first')]

    df = df.copy()
    df['DistrictName'] = joined['DistrictName'].reindex(df.index)
    matched = int(df['DistrictName'].notna().sum())
    print(f"  [{label}] DistrictName matched: {matched:,}/{len(df):,} "
          f"({matched/len(df)*100:.1f}%)")
    return df


try:
    import geopandas as gpd
    unified_districts = load_unified_districts()
    sold     = add_district_name(sold,     'Sold',    unified_districts)
    listings = add_district_name(listings, 'Listing', unified_districts)
except ImportError:
    print("  geopandas not installed. Run: pip install geopandas")
    sold['DistrictName']     = np.nan
    listings['DistrictName'] = np.nan
except Exception as e:
    print(f"  School district join failed: {e}")
    sold['DistrictName']     = np.nan
    listings['DistrictName'] = np.nan

# =========================================================================
# Part B — Market Metric Engineering
# Learned from teammate 1:
#   - safe_divide for all ratios
#   - compute_close_metrics flag to skip close-dependent metrics for listings
#   - Negative durations: kept as-is + _NegativeFlag column added
#   - PascalCase column names to match teammates
# =========================================================================

section("Part B: Market Metric Engineering")

def add_market_metrics(df, label, compute_close_metrics):
    df = df.copy()

    # PriceRatio = ClosePrice / ListPrice
    if compute_close_metrics and 'ClosePrice' in df.columns and 'ListPrice' in df.columns:
        df['PriceRatio'] = safe_divide(df['ClosePrice'], df['ListPrice'])
    else:
        print(f"  [{label}] Skipped PriceRatio (no ClosePrice for listings).")

    # CloseToOriginalListRatio = ClosePrice / OriginalListPrice
    if compute_close_metrics and 'ClosePrice' in df.columns and 'OriginalListPrice' in df.columns:
        df['CloseToOriginalListRatio'] = safe_divide(
            df['ClosePrice'], df['OriginalListPrice']
        )
        # Filter to plausible range (0.1 to 5.0)
        # Values outside this range are likely data entry errors
        # (e.g. OriginalListPrice recorded as $1 or $100)
        before = df['CloseToOriginalListRatio'].notna().sum()
        df['CloseToOriginalListRatio'] = df['CloseToOriginalListRatio'].where(
            df['CloseToOriginalListRatio'].between(0.1, 5.0), np.nan
        )
        after   = df['CloseToOriginalListRatio'].notna().sum()
        removed = before - after
        print(f"  [{label}] CloseToOriginalListRatio: "
              f"{removed:,} values outside [0.1, 5.0] set to NaN")

    # PricePerSqFt = ClosePrice / LivingArea
    if compute_close_metrics and 'ClosePrice' in df.columns and 'LivingArea' in df.columns:
        df['PricePerSqFt'] = safe_divide(df['ClosePrice'], df['LivingArea'])
    else:
        print(f"  [{label}] Skipped PricePerSqFt.")

    # DaysOnMarket — existing field, no computation needed

    # Year / Month / YrMo from CloseDate
    if compute_close_metrics and 'CloseDate' in df.columns:
        df['Year']  = df['CloseDate'].dt.year.astype('Int64')
        df['Month'] = df['CloseDate'].dt.month.astype('Int64')
        yrmo        = df['CloseDate'].dt.to_period('M').astype(str)
        df['YrMo']  = yrmo.where(df['CloseDate'].notna(), np.nan)
    else:
        print(f"  [{label}] Skipped Year/Month/YrMo.")

    # ListingToContractDays = PurchaseContractDate - ListingContractDate
    # Keep negative values (don't set to NaN) + add NegativeFlag column
    if 'PurchaseContractDate' in df.columns and 'ListingContractDate' in df.columns:
        df['ListingToContractDays'] = (
            df['PurchaseContractDate'] - df['ListingContractDate']
        ).dt.days
        df['ListingToContractDays_NegativeFlag'] = df['ListingToContractDays'] < 0
    else:
        print(f"  [{label}] Skipped ListingToContractDays.")

    # ContractToCloseDays = CloseDate - PurchaseContractDate
    if compute_close_metrics and 'CloseDate' in df.columns and 'PurchaseContractDate' in df.columns:
        df['ContractToCloseDays'] = (
            df['CloseDate'] - df['PurchaseContractDate']
        ).dt.days
        df['ContractToCloseDays_NegativeFlag'] = df['ContractToCloseDays'] < 0
    else:
        print(f"  [{label}] Skipped ContractToCloseDays.")

    return df


# Sold: all metrics computed (has ClosePrice / CloseDate)
# Listing: only ListingToContractDays computed (no ClosePrice / CloseDate)
sold     = add_market_metrics(sold,     'Sold',    compute_close_metrics=True)
listings = add_market_metrics(listings, 'Listing', compute_close_metrics=False)

# =========================================================================
# Validation (from teammate 1)
# =========================================================================

def validate_metrics(df, label):
    print(f"\n  --- [{label}] Metric Validation ---")

    source_cols = ['ClosePrice', 'ListPrice', 'OriginalListPrice',
                   'LivingArea', 'PurchaseContractDate',
                   'ListingContractDate', 'CloseDate']
    print("  Missing values in source columns:")
    for col in source_cols:
        if col in df.columns:
            n = int(df[col].isna().sum())
            print(f"    {col:<30} : {n:,} ({n/len(df)*100:.1f}%)")
        else:
            print(f"    {col:<30} : not present")

    print("  Zero/negative denominators (set to NaN in ratios):")
    for den in ['ListPrice', 'OriginalListPrice', 'LivingArea']:
        if den in df.columns:
            bad = int((df[den] <= 0).sum())
            print(f"    {den:<30} <= 0 : {bad:,}")

    print("  Negative date-duration flags:")
    for flag in ['ListingToContractDays_NegativeFlag',
                 'ContractToCloseDays_NegativeFlag']:
        if flag in df.columns:
            print(f"    {flag:<45} : {int(df[flag].sum()):,}")

    print("  Engineered column dtypes:")
    check = ['DistrictName', 'PriceRatio', 'CloseToOriginalListRatio',
             'PricePerSqFt', 'Year', 'Month', 'YrMo',
             'ListingToContractDays', 'ContractToCloseDays']
    for col in check:
        if col in df.columns:
            print(f"    {col:<35} : {df[col].dtype}")


validate_metrics(sold,     'Sold')
validate_metrics(listings, 'Listing')

# =========================================================================
# Sample output (from teammate 1)
# =========================================================================

def show_sample_output(df, label):
    cols = ['ListingKey', 'DistrictName', 'ClosePrice', 'ListPrice',
            'OriginalListPrice', 'PriceRatio', 'CloseToOriginalListRatio',
            'PricePerSqFt', 'DaysOnMarket', 'Year', 'Month', 'YrMo',
            'ListingToContractDays', 'ContractToCloseDays']
    available = [c for c in cols if c in df.columns]
    numeric   = [c for c in available if c not in ('ListingKey', 'DistrictName')]
    complete  = df.dropna(subset=numeric) if numeric else df
    sample    = (complete if len(complete) > 0 else df)[available].head(5)
    print(f"\n  --- [{label}] Sample Output ---")
    print(sample.to_string(index=False))


show_sample_output(sold,     'Sold')
show_sample_output(listings, 'Listing')

# =========================================================================
# Part C — Segment Analysis (Handbook required dimensions)
# =========================================================================

section("Part C: Segment Analysis")

def segment_summary(df, group_cols, label):
    """Handbook-required segment summary metrics."""
    valid_cols = [c for c in group_cols if c in df.columns]
    if not valid_cols:
        print(f"  [{label}] columns not found — skipped.")
        return None

    agg_map = {
        new_name: (src, how)
        for src, new_name, how in [
            ('ClosePrice',              'closed_sales',                      'count'),
            ('ClosePrice',              'median_close_price',                 'median'),
            ('PricePerSqFt',            'average_price_per_sqft',            'mean'),
            ('DaysOnMarket',            'average_days_on_market',            'mean'),
            ('PriceRatio',              'average_price_ratio',               'mean'),
            ('CloseToOriginalListRatio','average_close_to_original_list_ratio','mean'),
            ('ListingToContractDays',   'average_listing_to_contract_days',  'mean'),
            ('ContractToCloseDays',     'average_contract_to_close_days',    'mean'),
        ]
        if src in df.columns
    }

    grouped = df.groupby(valid_cols, dropna=False).agg(**agg_map).reset_index()
    grouped = grouped.sort_values('closed_sales', ascending=False)
    print(f"\n  --- {label} (top 10) ---")
    print(grouped.head(10).to_string(index=False))
    return grouped


sold_property_summary    = segment_summary(sold, ['PropertyType', 'PropertySubType'],
                                           'Sold — PropertyType x PropertySubType')
sold_geographic_summary  = segment_summary(sold, ['CountyOrParish', 'MLSAreaMajor'],
                                           'Sold — CountyOrParish x MLSAreaMajor')
sold_list_office_summary = segment_summary(sold, ['ListOfficeName'],
                                           'Sold — ListOfficeName')
sold_buyer_office_summary= segment_summary(sold, ['BuyerOfficeName'],
                                           'Sold — BuyerOfficeName')



# =========================================================================
# Save enriched datasets + validate exported structure
# =========================================================================

section("Save Outputs")

sold.to_csv(SOLD_OUTPUT,     index=False, encoding='utf-8')
listings.to_csv(LISTING_OUTPUT, index=False, encoding='utf-8')

# Validate (from teammate 1)
sold_check     = pd.read_csv(SOLD_OUTPUT,     nrows=0)
listings_check = pd.read_csv(LISTING_OUTPUT,  nrows=0)

print(f"\nSold enriched     : {SOLD_OUTPUT}")
print(f"  Shape: {sold.shape}")
print(f"List enriched     : {LISTING_OUTPUT}")
print(f"  Shape: {listings.shape}")
print(f"\nSold exported columns    : {len(sold_check.columns)}")
print(f"List exported columns    : {len(listings_check.columns)}")
print(f"DistrictName in Sold     : {'DistrictName' in sold_check.columns}")
print(f"DistrictName in Listing  : {'DistrictName' in listings_check.columns}")

print("\n" + "=" * 78)
print("WEEK 6 FEATURE ENGINEERING COMPLETE")
print("=" * 78)


Load Cleaned Data
Sold    : 447,769 rows  |  76 columns
Listing : 615,316 rows  |  67 columns

Part A: School District Enrichment
Loading cached Unified district boundaries...
Loaded 345 cached Unified district polygons.
  [Sold] DistrictName matched: 333,398/447,769 (74.5%)
  [Listing] DistrictName matched: 411,516/615,316 (66.9%)

Part B: Market Metric Engineering
  [Sold] CloseToOriginalListRatio: 959 values outside [0.1, 5.0] set to NaN
  [Listing] Skipped PriceRatio (no ClosePrice for listings).
  [Listing] Skipped PricePerSqFt.
  [Listing] Skipped Year/Month/YrMo.
  [Listing] Skipped ContractToCloseDays.

  --- [Sold] Metric Validation ---
  Missing values in source columns:
    ClosePrice                     : 2 (0.0%)
    ListPrice                      : 0 (0.0%)
    OriginalListPrice              : 822 (0.2%)
    LivingArea                     : 253 (0.1%)
    PurchaseContractDate           : 198 (0.0%)
    ListingContractDate            : 1 (0.0%)
    CloseDate              